# AIC2026 — Kaggle/Colab Notebook: Generate All Vector Embeddings (GPU)

**Mục đích:** Nạp dữ liệu thô hợp nhất (`scene_metadata.parquet`, `ocr_text.parquet`, `asr_text.parquet`, `keyframes/`), sử dụng **GPU (Kaggle/Colab)** để sinh đồng thời:
1. `image_vectors.npy` (1024-dim, BEiT3 + CLIP, L2-normalized)
2. `ocr_vectors.npy` (1536-dim, gte-Qwen2-1.5B, L2-normalized)
3. `asr_vectors.npy` (1536-dim, gte-Qwen2-1.5B, L2-normalized)
4. `scene_ids.txt` (khớp 1-to-1 thứ tự các hàng)

**Đầu ra:** File `unified_dataset.zip` tải về máy local để nạp vào Qdrant DB qua `python -m indexing.run_index`.

In [ ]:
# 1. Install dependencies
!pip install -q transformers pillow soundfile accelerate torch numpy pandas pyarrow pyyaml

import os
import zipfile
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone Private Repo (Nếu đã có folder AIC2026 thì bỏ qua)
# Nếu repo của bạn là Private, hãy nhập GitHub Token (PAT) bên dưới nếu cần clone tự động:
GITHUB_TOKEN = ""  # Nhập token của bạn dạng ghp_xxxx nếu cần

if not Path("AIC2026").exists():
    if GITHUB_TOKEN:
        repo_url = f"https://{GITHUB_TOKEN}@github.com/ThanhDanh1510/Super-Image-Retrieval-System-for-AIC2026.git"
    else:
        repo_url = "https://github.com/ThanhDanh1510/Super-Image-Retrieval-System-for-AIC2026.git"
    !git clone {repo_url} AIC2026

import sys
sys.path.append(str(Path("AIC2026").absolute()))

# Sử dụng importlib để import module có chữ số đứng đầu (01c_embedding_generation)
from shared.models.image_embedding_wrapper import ImageEmbeddingWrapper
from shared.models.text_embedding_wrapper import TextEmbeddingWrapper

embed_mod = importlib.import_module("ingestion.skeletons.01c_embedding_generation")
run_embedding_generation = embed_mod.run_embedding_generation
print("✅ Imported code modules successfully!")

In [ ]:
# 3. Thiết lập đường dẫn Input & Output
# Hỗ trợ tự động nhận diện cả Kaggle (/kaggle/input) lẫn Google Colab (/content)
OUTPUT_DIR = Path("./unified_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_dir = None
possible_paths = [
    Path("/kaggle/input/aic2026-unified-raw"),
    Path("./unified_raw"),
    Path("/content/unified_raw"),
]

for p in possible_paths:
    if p.exists():
        input_dir = p
        break

if not input_dir and Path("/kaggle/input").exists():
    inputs = list(Path("/kaggle/input").glob("*"))
    if inputs:
        input_dir = inputs[0]

print(f"INPUT DIR: {input_dir}")
assert input_dir and input_dir.exists(), "Không tìm thấy thư mục input chứa parquets và keyframes!"

In [ ]:
# 4. Chạy Pipeline sinh Embeddings bằng GPU
metadata_path = str(input_dir / "scene_metadata.parquet")
ocr_path = str(input_dir / "ocr_text.parquet")
asr_path = str(input_dir / "asr_text.parquet")
keyframe_dir = str(input_dir / "keyframes")

run_embedding_generation(
    metadata_path=metadata_path,
    ocr_path=ocr_path,
    asr_path=asr_path,
    keyframe_dir=keyframe_dir,
    output_dir=str(OUTPUT_DIR),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [ ]:
# 5. Copy parquets và đóng gói thành unified_dataset.zip cho local download
import shutil

shutil.copy2(metadata_path, OUTPUT_DIR / "scene_metadata.parquet")
shutil.copy2(ocr_path, OUTPUT_DIR / "ocr_text.parquet")
shutil.copy2(asr_path, OUTPUT_DIR / "asr_text.parquet")

zip_filename = "./unified_dataset.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            zipf.write(file, arcname=file.relative_to(OUTPUT_DIR))

print(f"✅ ĐÃ HOÀN THÀNH VÀ ĐÓNG GÓI: {zip_filename}")
print(f"Kích thước file zip: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")